In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/cs-training.csv").drop(columns=["Unnamed: 0"])
df = df.rename(columns={
    "SeriousDlqin2yrs": "default_2yrs",
    "RevolvingUtilizationOfUnsecuredLines": "utilisation",
    "NumberOfTime30-59DaysPastDueNotWorse": "late_30_59",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_lines",
    "NumberOfTimes90DaysLate": "late_90",
    "NumberRealEstateLoansOrLines": "real_estate_loans",
    "NumberOfTime60-89DaysPastDueNotWorse": "late_60_89",
    "NumberOfDependents": "dependents"})
df.columns.tolist()

['default_2yrs',
 'utilisation',
 'age',
 'late_30_59',
 'debt_ratio',
 'monthly_income',
 'open_credit_lines',
 'late_90',
 'real_estate_loans',
 'late_60_89',
 'dependents']

In [2]:
late_cols = ["late_30_59", "late_60_89", "late_90"]
print("Age 0:", (df["age"] == 0).sum())
print("Rows with 96/98 codes:", df[late_cols].isin([96, 98]).any(axis=1).sum())
print("Utilisation > 10:", (df["utilisation"] > 10).sum())
print("Utilisation 1–10 (over limit):", df["utilisation"].between(1, 10, inclusive="right").sum())
print("Income missing:", df["monthly_income"].isna().sum())
print("Dependents missing:", df["dependents"].isna().sum())

Age 0: 1
Rows with 96/98 codes: 269
Utilisation > 10: 241
Utilisation 1–10 (over limit): 3080
Income missing: 29731
Dependents missing: 3924


In [3]:
df = df[df["age"] > 0].copy()
df[late_cols] = df[late_cols].replace([96, 98], np.nan)
df.loc[df["utilisation"] > 10, "utilisation"] = np.nan
df.loc[df["monthly_income"].isna(), "debt_ratio"] = np.nan
df.isna().sum()

default_2yrs             0
utilisation            241
age                      0
late_30_59             269
debt_ratio           29731
monthly_income       29731
open_credit_lines        0
late_90                269
real_estate_loans        0
late_60_89             269
dependents            3924
dtype: int64

In [4]:
from sklearn.model_selection import train_test_split
import os

train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df["default_2yrs"])
print("Train:", train.shape, "default rate:", round(train["default_2yrs"].mean(), 4))
print("Test: ", test.shape, "default rate:", round(test["default_2yrs"].mean(), 4))

os.makedirs("data/clean", exist_ok=True)
train.to_csv("data/clean/train.csv", index=False)
test.to_csv("data/clean/test.csv", index=False)

Train: (104999, 11) default rate: 0.0668
Test:  (45000, 11) default rate: 0.0668
